# Stage 4 — Phase 1 Training (All Folds)

## RUN INSTRUCTIONS
1. **Run this notebook first** (Phase 1 — frozen backbone, all 5 folds)
2. Wait for completion — 5 checkpoints saved as `fold_{k}_p1_best.h5`
3. **Restart kernel** (fully resets VRAM)
4. Open and run `phase2_finetune_all_folds.ipynb`

**What this does:** Trains the custom head on top of a fully frozen VGG-Face backbone for each fold.  
No backbone gradients → low VRAM, safe on 6 GB.

In [1]:
import os
import numpy as np
import gc
import warnings
warnings.filterwarnings('ignore')

# ── GPU memory growth — MUST come before any TF/DeepFace call ────────────────
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'Memory growth enabled on {len(gpus)} GPU(s)')

# ── Mixed precision — set ONCE here, never inside the fold loop ───────────────
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f'Compute policy: {tf.keras.mixed_precision.global_policy().name}')

from tensorflow.keras.layers import (
    Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import accuracy_score, precision_score, recall_score
from deepface import DeepFace
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f'TensorFlow: {tf.__version__}')
print(f'GPU devices: {len(gpus)}')

Memory growth enabled on 1 GPU(s)
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU, compute capability 8.6
Compute policy: mixed_float16
TensorFlow: 2.10.0
GPU devices: 1


## Configuration

In [2]:
DATA_DIR     = r'C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\data\imgData\prepImg\step4'
STAGE4_DIR   = r'C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4'
MODELS_DIR   = os.path.join(STAGE4_DIR, 'models')
FOLDS_DIR    = os.path.join(STAGE4_DIR, 'Folds')
PLOTS_DIR    = os.path.join(STAGE4_DIR, 'plots')
DATA_OUT_DIR = os.path.join(STAGE4_DIR, 'data')

for d in [MODELS_DIR, FOLDS_DIR, PLOTS_DIR, DATA_OUT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'Ready: {d}')

SEED         = 123
IMG_SIZE     = (224, 224)
NUM_FOLDS    = 5
BATCH_PHASE1 = 16
LR_PHASE1    = 1e-4
EPOCHS_P1    = 35
CLASS_WEIGHT = {0: 1.0, 1: 2.0}
VGG_MEAN     = [93.5940, 104.7624, 129.1863]
CLASS_NAMES  = ['Autistic', 'Non_Autistic']

print(f'\nBatch: {BATCH_PHASE1} | LR: {LR_PHASE1} | Epochs: {EPOCHS_P1}')
print(f'Class weights: {CLASS_WEIGHT}')

Ready: C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\models
Ready: C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\Folds
Ready: C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\plots
Ready: C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\data

Batch: 16 | LR: 0.0001 | Epochs: 35
Class weights: {0: 1.0, 1: 2.0}


## Helper Functions

In [3]:
def build_stage4_model(backbone):
    """Identical architecture to ML1.ipynb — DO NOT modify."""
    inputs = Input(shape=(224, 224, 3), name='face_input')
    x = backbone(inputs, training=False)
    x = GlobalAveragePooling2D(name='gap')(x)
    x = Dense(512, activation='relu',
               kernel_regularizer=l2(0.001),
               name='asd_feature_vector_512')(x)
    x = BatchNormalization(name='bn_512')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu',
               kernel_regularizer=l2(0.001),
               name='asd_feature_vector_256')(x)
    x = Dropout(0.4)(x)
    output = Dense(1, activation='sigmoid', name='classification_output',
                   dtype='float32')(x)
    return Model(inputs=inputs, outputs=output, name='ASD_Stage4')

print('build_stage4_model defined.')

build_stage4_model defined.


## Phase 1 — 5-Fold Training Loop

Backbone is **fully frozen**. Only the custom head is trained.  
Best checkpoint per fold saved as `fold_{k}_p1_best.h5`.

In [4]:
for fold_id in range(1, NUM_FOLDS + 1):
    print('\n' + '='*70)
    print(f'  FOLD {fold_id} / {NUM_FOLDS}  —  Phase 1 (frozen backbone)')
    print('='*70)

    # ── Load fold splits ──────────────────────────────────────────────────
    train_paths_fold  = np.load(os.path.join(FOLDS_DIR, f'fold_{fold_id}_train_paths.npy'),  allow_pickle=True)
    train_labels_fold = np.load(os.path.join(FOLDS_DIR, f'fold_{fold_id}_train_labels.npy'), allow_pickle=True)
    val_paths_fold    = np.load(os.path.join(FOLDS_DIR, f'fold_{fold_id}_val_paths.npy'),    allow_pickle=True)
    val_labels_fold   = np.load(os.path.join(FOLDS_DIR, f'fold_{fold_id}_val_labels.npy'),   allow_pickle=True)
    print(f'  Train: {len(train_paths_fold)}  Val: {len(val_paths_fold)}')

    # ── Clear session (do NOT call set_global_policy here) ────────────────
    tf.keras.backend.clear_session()
    gc.collect()

    # ── Rebuild augmentation fresh after clear_session ────────────────────
    data_augmentation_fold = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.15),
        tf.keras.layers.RandomContrast(0.1),
        tf.keras.layers.RandomZoom(0.1),
    ], name='augmentation')

    def process_image_fold(file_path, label, augment=False):
        img = tf.io.read_file(file_path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = img[..., ::-1]                        # RGB → BGR
        img = tf.cast(img, tf.float32) - VGG_MEAN  # subtract channel means
        if augment:
            img = tf.expand_dims(img, 0)
            img = data_augmentation_fold(img, training=True)
            img = tf.squeeze(img, 0)
        target = 1.0 - tf.cast(label, tf.float32)  # Autistic=1
        return img, target

    def make_ds(paths, labels, batch_size, augment=False, shuffle=False):
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if shuffle:
            ds = ds.shuffle(len(paths), seed=SEED)
        ds = ds.map(lambda x, y: process_image_fold(x, y, augment=augment),
                    num_parallel_calls=tf.data.AUTOTUNE)
        return ds.batch(batch_size).prefetch(1)  # prefetch(1) avoids AUTOTUNE over-buffering

    # ── Build datasets ────────────────────────────────────────────────────
    train_ds = make_ds(train_paths_fold, train_labels_fold,
                       BATCH_PHASE1, augment=True, shuffle=True)
    val_ds   = make_ds(val_paths_fold, val_labels_fold,
                       BATCH_PHASE1, augment=False, shuffle=False)

    # ── Build backbone + model ────────────────────────────────────────────
    vgg_wrapper  = DeepFace.build_model('VGG-Face')
    full_vgg     = vgg_wrapper.model
    backbone_out = full_vgg.get_layer('conv2d_14').output
    backbone     = Model(inputs=full_vgg.input, outputs=backbone_out,
                         name='vgg_face_backbone')
    backbone.trainable = False
    model = build_stage4_model(backbone)

    trainable_params = sum(np.prod(w.shape) for w in model.trainable_weights)
    print(f'  Trainable params (head only): {trainable_params:,}')

    # ── Callbacks ─────────────────────────────────────────────────────────
    p1_path = os.path.join(MODELS_DIR, f'fold_{fold_id}_p1_best.h5')
    callbacks_p1 = [
        EarlyStopping(monitor='val_loss', patience=8,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                          verbose=1, min_lr=1e-7),
        ModelCheckpoint(p1_path, monitor='val_loss',
                        save_best_only=True, mode='min', verbose=1),
    ]

    # ── Phase 1 training ──────────────────────────────────────────────────
    print(f'\n  Phase 1 — frozen backbone  (LR={LR_PHASE1}, batch={BATCH_PHASE1})')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_PHASE1),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    history_p1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_P1,
        class_weight=CLASS_WEIGHT,
        callbacks=callbacks_p1,
        verbose=1
    )
    print(f'  Checkpoint saved: {p1_path}')

    # ── Training plot ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Fold {fold_id} — Phase 1 Training History', fontsize=13)
    ep = range(1, len(history_p1.history['loss']) + 1)
    axes[0].plot(ep, history_p1.history['loss'],        'b-',  label='train')
    axes[0].plot(ep, history_p1.history['val_loss'],    'b--', label='val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(ep, history_p1.history['accuracy'],     'b-',  label='train')
    axes[1].plot(ep, history_p1.history['val_accuracy'], 'b--', label='val')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, f'fold_{fold_id}_p1_history.png')
    plt.savefig(plot_path, dpi=100, bbox_inches='tight')
    plt.close()
    print(f'  Plot saved: {plot_path}')

    # ── Full cleanup ───────────────────────────────────────────────────────
    del model, backbone, vgg_wrapper, full_vgg, backbone_out
    del train_ds, val_ds
    del history_p1
    gc.collect()
    tf.keras.backend.clear_session()
    print(f'  Fold {fold_id} cleanup done.')

print('\n' + '='*70)
print('  PHASE 1 COMPLETE — all 5 folds trained.')
print('  Next: Restart kernel, then run phase2_finetune_all_folds.ipynb')
print('='*70)


  FOLD 1 / 5  —  Phase 1 (frozen backbone)
  Train: 2820  Val: 705
  Trainable params (head only): 2,230,273

  Phase 1 — frozen backbone  (LR=0.0001, batch=16)
Epoch 1/35
177/177 [==============================] - ETA: 0s - loss: 2.4313 - accuracy: 0.5447 - precision: 0.5313 - recall: 0.7310
Epoch 1: val_loss improved from inf to 1.86910, saving model to C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\models\fold_1_p1_best.h5
177/177 [==============================] - 43s 201ms/step - loss: 2.4313 - accuracy: 0.5447 - precision: 0.5313 - recall: 0.7310 - val_loss: 1.8691 - val_accuracy: 0.6582 - val_precision: 0.7004 - val_recall: 0.5511 - lr: 1.0000e-04
Epoch 2/35
177/177 [==============================] - ETA: 0s - loss: 2.2272 - accuracy: 0.6082 - precision: 0.5776 - recall: 0.7950
Epoch 2: val_loss improved from 1.86910 to 1.83187, saving model to C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project